# EEG_15 — DHSLP (Li et al. 2025): replica fedele

**Riferimento**: Li et al. 2025 — "EEG-based speech imagery decoding by dynamic hypergraph learning within projected and selected feature subspaces", J. Neural Eng. 22 (2025) 046030.

**Architettura**:
- Vertici = trial (ogni campione EEG = nodo)
- Feature per vertice: 12 temporal features da banda gamma → selezione top-2 per PCC → 2×61ch = 122D
- Iperedge: distanza-based, **una iperedge per vertice** → |V| = |E|: eᵢ = {vⱼ | d(vᵢ,vⱼ) ≤ ξ·d̄ᵢ}
- Semi-supervised per soggetto: labeled = sess.1-3 (train), unlabeled = sess.4 (val per LP)
- Test: sess.5, classificato con **1-NN nel sottospazio proiettato M**
- Ottimizzazione: **Algorithm 1** — alternating su {Fu, M, H, U, W}

**Differenza da EEG_13**: EEG_13 = elettrodi come vertici, H appresa end-to-end.  
**DHSLP qui** = trial come vertici, H costruita da distanze, semi-supervised label propagation.

| Variabile | Significato |
|-----------|-------------|
| X ∈ R^{n×d} | feature matrix (train + val per soggetto) |
| H ∈ {0,1}^{n×n} | incidence matrix (una iperedge per trial) |
| W ∈ R^{n×n} | pesi iperedge (diagonale) |
| U ∈ R^{n×n} | pesi vertici (diagonale) |
| M ∈ R^{d×f} | projection matrix DHSLP |
| Fu ∈ R^{u×c} | pseudo-label per campioni unlabeled (val) |


In [ ]:
import json, logging, re, time
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import wandb
from scipy import signal as scipy_signal
from scipy.linalg import eigh
from scipy.stats import pearsonr
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('eeg15')

project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents)
                     if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_DIR = project_root / 'models' / 'eeg15'; CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_CHANNELS    = 61
N_SAMPLES     = 384
SFREQ         = 256
N_CLASSES     = 4
CLUSTER_SCHEME = 'concr4'

# Feature extraction (Li et al. §4.1)
GAMMA_LOW  = 30    # Hz
GAMMA_HIGH = 100   # Hz
N_TEMP_FEATURES = 12  # temporal features da gamma band per canale
N_SELECT_FEAT   = 2   # top-N per PCC selezionati → 2×61=122D finale

# DHSLP iperparametri (grid search come Li et al.)
XI_LIST    = [0.3, 0.5, 0.7]         # hyperedge connection parameter
ALPHA_LIST = [2**k for k in (-4,-2,0,2,4)]   # smoothness weight
BETA_LIST  = [2**k for k in (-4,-2,0,2,4)]   # W regularization weight
F_DIM_LIST = [20, 40, 60]            # projected subspace dimensionality
MAX_ITER   = 30                       # iterazioni massime Algorithm 1

# Split per soggetto (identico a EEG_06 subject-specific)
TRAIN_SESSIONS = [1, 2, 3]   # labeled
VAL_SESSIONS   = [4]         # unlabeled (per semi-supervised)
TEST_SESSIONS  = [5]         # test (classificato con 1-NN in subspace)

# Sorgente dati
DATA_METRIC   = 'abs_pcc'
WANDB_ENTITY  = 'uras-daniele22-politecnico-di-milano'
WANDB_PROJECT = 'miralis-imagined-speech'

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
label2cluster = {int(k): int(v) for k,v in json.loads(
    (project_root/'configs'/'label_schemes'/'labelid2cluster_concr4.json').read_text()).items()}

HG_ROOT = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
subj_sess = defaultdict(lambda: defaultdict(list))
for p in sorted(HG_ROOT.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m:
        subj_sess[int(m.group(1))][int(m.group(2))].append(p)

ALL_SUBJ = sorted(subj_sess.keys())
# Solo soggetti con tutte le sessioni 1-5
VALID_SUBJ = [s for s in ALL_SUBJ if all(
    ses in subj_sess[s] for ses in TRAIN_SESSIONS + VAL_SESSIONS + TEST_SESSIONS)]
log.info(f'Soggetti totali: {len(ALL_SUBJ)}  con tutte le sessioni 1-5: {len(VALID_SUBJ)}')


## §2 — Feature Extraction (12 Temporal Features da Gamma)

Li et al. §4.1: gamma-band filtering → 12 temporal features per canale → selezione top-2 per PCC con label → feature vector = 2×61 = 122D per trial.

Le 12 feature temporali (Table 2 del paper):
| # | Feature | Descrizione |
|---|---------|-------------|
| T1 | Centroid (time axis) | Baricentro temporale pesato da |x(t)| |
| T2 | Mean absolute difference | mean(|x[t+1]-x[t]|) |
| T3 | Mean difference | mean(x[t+1]-x[t]) |
| T4 | Median absolute difference | median(|x[t+1]-x[t]|) |
| T5 | Median difference | median(x[t+1]-x[t]) |
| T6 | Signal traveled distance | sum(sqrt(1+(x[t+1]-x[t])^2)) |
| T7 | Sum of absolute difference | sum(|x[t+1]-x[t]|) |
| T8 | Slope | pendenza regressione lineare |
| T9 | Area under curve | sum(|x(t)|)/SFREQ |
| T10 | Peak to peak | max(x) - min(x) |
| T11 | Shannon entropy | -sum(p*log2(p)) da istogramma |
| T12 | Neighborhood peaks | numero massimi locali |


In [ ]:
# Butterworth bandpass gamma
_b_g, _a_g = scipy_signal.butter(4, [GAMMA_LOW, GAMMA_HIGH], btype='bandpass', fs=SFREQ)

def _shannon_entropy(sig, n_bins=16):
    counts, _ = np.histogram(sig, bins=n_bins)
    probs = counts / (counts.sum() + 1e-12)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))

def extract_12_features(x_ch):
    """
    x_ch: (T,) segnale gamma-filtrato per un canale.
    Output: (12,) float
    """
    T = len(x_ch)
    t = np.arange(T, dtype=np.float32)
    diff = np.diff(x_ch)   # (T-1,)
    abs_diff = np.abs(diff)

    T1 = np.sum(t * np.abs(x_ch)) / (np.sum(np.abs(x_ch)) + 1e-12)
    T2 = np.mean(abs_diff)
    T3 = np.mean(diff)
    T4 = np.median(abs_diff)
    T5 = np.median(diff)
    T6 = np.sum(np.sqrt(1.0 + diff**2))
    T7 = np.sum(abs_diff)
    coefs = np.polyfit(t, x_ch, 1)
    T8 = coefs[0]
    T9 = np.sum(np.abs(x_ch)) / SFREQ
    T10 = np.max(x_ch) - np.min(x_ch)
    T11 = _shannon_entropy(x_ch)
    # Local maxima count
    loc_max = np.sum((x_ch[1:-1] > x_ch[:-2]) & (x_ch[1:-1] > x_ch[2:]))
    T12 = float(loc_max)

    return np.array([T1,T2,T3,T4,T5,T6,T7,T8,T9,T10,T11,T12], dtype=np.float32)

def extract_trial_features(x_np):
    """
    x_np: (61, 384)
    Output: (61, 12) — una riga per canale, 12 feature temporali da gamma
    """
    x_filt = scipy_signal.filtfilt(_b_g, _a_g, x_np, axis=1)  # (61, 384)
    feats = np.stack([extract_12_features(x_filt[c]) for c in range(N_CHANNELS)])
    return feats  # (61, 12)


FEAT_CACHE = FIG_DIR / 'eeg15v2_trial_raw12feat.npz'

if FEAT_CACHE.exists():
    log.info(f'Carico feature da cache: {FEAT_CACHE}')
    cache = np.load(FEAT_CACHE, allow_pickle=True)
    RAW12_FEATURES = cache['features']   # (N_trials, 61, 12)
    RAW12_LABELS   = cache['labels'].astype(int)
    RAW12_SUBJ     = cache['subjects'].astype(int)
    RAW12_SESS     = cache['sessions'].astype(int)
else:
    log.info('Estrazione 12 feature gamma... (~5-10 min)')
    feat_list, lbl_list, subj_list, sess_list = [], [], [], []
    for sid in tqdm(VALID_SUBJ, desc='Soggetti'):
        for ses in TRAIN_SESSIONS + VAL_SESSIONS + TEST_SESSIONS:
            for p in subj_sess[sid].get(ses, []):
                try:
                    d = torch.load(p, weights_only=False)
                    x_np = d['x'].float().numpy()
                    y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
                    c = label2cluster.get(y_word)
                    if c is None: continue
                    feats = extract_trial_features(x_np)   # (61, 12)
                    feat_list.append(feats)
                    lbl_list.append(c)
                    subj_list.append(sid)
                    sess_list.append(ses)
                except Exception as e:
                    log.warning(f'{p}: {e}')
    RAW12_FEATURES = np.stack(feat_list, axis=0).astype(np.float32)
    RAW12_LABELS   = np.array(lbl_list,  dtype=np.int32)
    RAW12_SUBJ     = np.array(subj_list, dtype=np.int32)
    RAW12_SESS     = np.array(sess_list, dtype=np.int32)
    np.savez(FEAT_CACHE, features=RAW12_FEATURES, labels=RAW12_LABELS,
             subjects=RAW12_SUBJ, sessions=RAW12_SESS)
    log.info(f'Cache salvata: {RAW12_FEATURES.shape}')

N_TOTAL = len(RAW12_FEATURES)
log.info(f'Totale trial: {N_TOTAL}')
log.info(f'Shape feature: {RAW12_FEATURES.shape}  (trials, canali, 12 features)')


## §3 — Selezione Feature per PCC (Top-2)

Per ogni feature temporale (tra le 12): calcola PCC medio con le label (one-vs-rest, media tra classi).  
Seleziona le `N_SELECT_FEAT=2` feature con PCC assoluto più alto → **feature matrix finale** (N_trials, 61×2=122).

La selezione è fit sul training set di ogni soggetto, ma poiché è analysis globale la eseguiamo una volta su tutti i trial di tutti i soggetti training.


In [ ]:
def select_features_by_pcc(feats_12, labels, n_select=N_SELECT_FEAT):
    """
    feats_12: (N, 61, 12)
    labels:   (N,) int
    Ritorna: (12,) bool mask delle feature selezionate (top n_select per |PCC|)
    """
    # Per ogni feature f∈[0..11]: PCC medio |r| tra feats[:,ch,f] e label (one-vs-rest)
    pcc_scores = np.zeros(12, dtype=np.float32)
    for f in range(12):
        r_sum = 0.0
        count = 0
        for ch in range(N_CHANNELS):
            feat_vec = feats_12[:, ch, f]
            for cls in range(N_CLASSES):
                y_bin = (labels == cls).astype(np.float32)
                r, _ = pearsonr(feat_vec, y_bin)
                if not np.isnan(r):
                    r_sum += abs(r)
                    count += 1
        pcc_scores[f] = r_sum / max(count, 1)
    top_idx = np.argsort(pcc_scores)[-n_select:]
    return top_idx, pcc_scores

# Calcola PCC su tutti i trial delle sessioni di training
train_mask_global = np.isin(RAW12_SESS, TRAIN_SESSIONS)
feat_train_global = RAW12_FEATURES[train_mask_global]
lbl_train_global  = RAW12_LABELS[train_mask_global]

log.info('Calcolo PCC per selezione feature (su tutti i trial training)...')
TOP_FEAT_IDX, PCC_SCORES = select_features_by_pcc(feat_train_global, lbl_train_global)

FEAT_NAMES_12 = ['T1_centroid','T2_mean_absdiff','T3_mean_diff',
                 'T4_med_absdiff','T5_med_diff','T6_travel_dist',
                 'T7_sum_absdiff','T8_slope','T9_area','T10_ptp',
                 'T11_entropy','T12_peaks']
log.info(f'PCC scores: {dict(zip(FEAT_NAMES_12, PCC_SCORES.round(4)))}')
log.info(f'Top-{N_SELECT_FEAT} feature selezionate: {[FEAT_NAMES_12[i] for i in TOP_FEAT_IDX]}')

def build_feature_vector(feats_12_trial):
    """feats_12_trial: (61, 12) → (61*N_SELECT_FEAT,) con le feature selezionate"""
    return feats_12_trial[:, TOP_FEAT_IDX].flatten()  # (61*2=122,)

# Feature matrix finale per tutti i trial
ALL_FEATURES_122 = np.stack([build_feature_vector(RAW12_FEATURES[i])
                              for i in range(N_TOTAL)], axis=0)  # (N, 122)

log.info(f'Feature matrix finale: {ALL_FEATURES_122.shape}  ({N_TOTAL} trial × {ALL_FEATURES_122.shape[1]}D)')

# Plot PCC scores
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2C7BB6' if i in TOP_FEAT_IDX else '#CCCCCC' for i in range(12)]
ax.bar(FEAT_NAMES_12, PCC_SCORES, color=colors, alpha=0.9)
ax.set_title(f'PCC Features vs Label — top-{N_SELECT_FEAT} selezionate (blu)')
ax.set_ylabel('|PCC| medio'); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg15_pcc_feature_selection.png', dpi=150); plt.show()


## §4 — DHSLP: Algorithm 1 (Li et al.)

Implementazione dell'ottimizzazione alternante per DHSLP:

```
Input: X_l (labeled, n_l×d), Y_l (n_l×c), X_u (unlabeled, n_u×d), params ξ, α, β, f
Init:  H (distance-based), W=I/n, U (distance-based)
Loop:
  1. Fu = -Luu^{-1} Lul Yl
  2. M = f eigenvettori più piccoli di α·X^T·L·X
  3. H, U = costruzione distanza nel sottospazio M
  4. W = Newton+Lagrange su w·Dv/De
Output: M, Fu — test classificato con 1-NN(M·X_train, M·X_test)
```


In [ ]:
def build_H_distance(X, xi):
    """
    Costruisce H (n×n) distanza-based: per ogni vi, iperedge ei = {vj | d(vi,vj) ≤ ξ·d̄i}
    X: (n, d)  — feature matrix nel sottospazio corrente
    Ritorna H sparsa (n×n) come array bool.
    """
    n = len(X)
    # Pairwise distances (euclidea)
    # Per n piccolo (≤500 per soggetto) è fattibile dense
    diff = X[:, None, :] - X[None, :, :]   # (n, n, d)
    D = np.sqrt((diff**2).sum(axis=2))     # (n, n)
    d_avg = D.mean(axis=1)                 # (n,) — d̄i

    H = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        threshold = xi * d_avg[i]
        H[i, :] = (D[i] <= threshold).astype(np.float32)
    return H, D


def compute_vertex_weights(D):
    """U(vi) = d̄i / sum(d̄j) — peso proporzionale alla distanza media"""
    d_avg = D.mean(axis=1)
    u = d_avg / (d_avg.sum() + 1e-12)
    return np.diag(u)


def compute_laplacian(H, W_diag, U):
    """
    L = U - Dv^{-1/2} U H W De^{-1} H^T U Dv^{-1/2}
    H: (n,n) incidence  W_diag: (n,) pesi iperedge  U: (n,n) pesi vertici
    """
    n = H.shape[0]
    W = np.diag(W_diag)

    Dv = np.diag(H @ W_diag)            # (n,n) — degree vertici
    De_diag = (H.T @ np.diag(U).reshape(-1,1)).flatten()  # (n,) — degree iperedge
    De_inv = np.diag(1.0 / np.maximum(De_diag, 1e-12))

    dv_diag = np.diag(Dv)
    Dv_invsqrt = np.diag(1.0 / np.maximum(np.sqrt(dv_diag), 1e-12))

    L = U - Dv_invsqrt @ U @ H @ W @ De_inv @ H.T @ U @ Dv_invsqrt
    return L


def update_W(H, W_diag, U, F, X, M, alpha, beta):
    """
    Update W via Newton+Lagrange:
    min -p·w - α·q·w + β·‖w‖² s.t. sum(w)=1, w>0
    Closed form via Lagrange: wi = (pi + α·qi)/(2β) (normalizzato)
    """
    n = H.shape[0]
    Dv = np.diag(H @ W_diag)
    dv = np.diag(Dv)
    Dv_invsqrt = np.diag(1.0 / np.maximum(np.sqrt(dv), 1e-12))
    De_diag = (H.T @ np.diag(U).reshape(-1,1)).flatten()
    De_inv = np.diag(1.0 / np.maximum(De_diag, 1e-12))

    # P = De^{-1} H^T U Dv^{-1/2} F F^T Dv^{-1/2} U H → diag
    inner_F = Dv_invsqrt @ U @ F
    P_mat = De_inv @ H.T @ U @ inner_F @ inner_F.T @ U @ H
    p = np.diag(P_mat)

    # Q = De^{-1} H^T U Dv^{-1/2} X M M^T X^T Dv^{-1/2} U H → diag
    XM = X @ M
    inner_Q = Dv_invsqrt @ U @ XM
    Q_mat = De_inv @ H.T @ U @ inner_Q @ inner_Q.T @ U @ H
    q = np.diag(Q_mat)

    # Newton+Lagrange: wi = (pi + α·qi)/(2β), poi normalize to sum=1
    raw = (p + alpha * q) / (2 * beta + 1e-12)
    raw = np.maximum(raw, 1e-12)
    w_new = raw / (raw.sum() + 1e-12)
    return w_new


def dhslp_subject(X_l, Y_l_onehot, X_u, f_dim, xi, alpha, beta, max_iter=MAX_ITER):
    """
    DHSLP Algorithm 1 per un soggetto.
    X_l: (n_l, d)  Y_l_onehot: (n_l, c)  X_u: (n_u, d)
    Ritorna: Fu (n_u, c), M (d, f)
    """
    n_l, d = X_l.shape
    n_u    = X_u.shape[0]
    n      = n_l + n_u
    c      = Y_l_onehot.shape[1]

    X = np.vstack([X_l, X_u])   # (n, d)

    # Label matrix con zeri per unlabeled
    F = np.zeros((n, c), dtype=np.float64)
    F[:n_l] = Y_l_onehot

    # Init: hypergraph in original space
    H, D  = build_H_distance(X, xi)
    U     = compute_vertex_weights(D)
    W_diag = np.ones(n, dtype=np.float64) / n

    # Init M: identità ridotta
    M = np.eye(d, min(f_dim, d), dtype=np.float64)

    for it in range(max_iter):
        # 1. Calcola Laplaciano
        L = compute_laplacian(H, W_diag, np.diag(U))

        # 2. Update Fu: Fu = -Luu^{-1} Lul Yl
        Lll = L[:n_l, :n_l]
        Llu = L[:n_l, n_l:]
        Lul = L[n_l:, :n_l]
        Luu = L[n_l:, n_l:]

        try:
            Fu = -np.linalg.solve(Luu + 1e-8*np.eye(n_u), Lul @ Y_l_onehot)
        except np.linalg.LinAlgError:
            Fu = np.zeros((n_u, c))
        F[n_l:] = Fu

        # 3. Update M: f eigenvettori più piccoli di α·X^T L X
        XLX = alpha * (X.T @ L @ X)
        try:
            eigenvalues, eigenvectors = eigh(XLX)
            M = eigenvectors[:, :f_dim]   # (d, f)
        except Exception:
            pass

        # 4. Update H, U nel sottospazio M
        X_proj = X @ M   # (n, f)
        H, D   = build_H_distance(X_proj, xi)
        U      = compute_vertex_weights(D)

        # 5. Update W
        W_diag = update_W(H, W_diag, np.diag(U), F, X, M, alpha, beta)

    return Fu, M


def classify_test(X_l, Y_l_int, X_te, M, k=1):
    """1-NN nel sottospazio M proiettato"""
    X_l_proj  = X_l  @ M
    X_te_proj = X_te @ M
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn.fit(X_l_proj, Y_l_int)
    return knn.predict(X_te_proj)


log.info('DHSLP implementato — pronto per il training per soggetto')


## §5 — Loop Per-Soggetto + Grid Search

Per ogni soggetto valido:
1. Split: sess.1-3 labeled, sess.4 unlabeled (per LP), sess.5 test
2. Grid search su (ξ, α, β, f) — val_bacc per scegliere config migliore
3. Retrain su configurazione ottimale → test_bacc su sess.5


In [ ]:
SUBJECT_RESULTS_15 = {}
FEAT_CACHE2 = CKPT_DIR / 'eeg15_subject_results.npz'

if FEAT_CACHE2.exists():
    log.info(f'Ricarico risultati da cache: {FEAT_CACHE2}')
    cache2 = np.load(FEAT_CACHE2, allow_pickle=True)
    SUBJECT_RESULTS_15 = cache2['results'].item()
else:
    for sid in tqdm(VALID_SUBJ[:5], desc='Soggetti DHSLP'):  # primi 5 per test — rimuovi [:5] per tutti
        # Raccogli trial per sessione
        mask_tr = (RAW12_SUBJ == sid) & np.isin(RAW12_SESS, TRAIN_SESSIONS)
        mask_va = (RAW12_SUBJ == sid) & np.isin(RAW12_SESS, VAL_SESSIONS)
        mask_te = (RAW12_SUBJ == sid) & np.isin(RAW12_SESS, TEST_SESSIONS)

        if mask_tr.sum() < 10 or mask_te.sum() < 5:
            log.warning(f'P{sid:03d}: dati insufficienti, skip')
            continue

        X_l  = ALL_FEATURES_122[mask_tr].astype(np.float64)
        Y_l  = RAW12_LABELS[mask_tr]
        X_u  = ALL_FEATURES_122[mask_va].astype(np.float64)
        X_te = ALL_FEATURES_122[mask_te].astype(np.float64)
        Y_te = RAW12_LABELS[mask_te]

        # One-hot per Y_l
        Y_l_oh = np.eye(N_CLASSES, dtype=np.float64)[Y_l]

        # Normalizzazione (fit su train)
        mu  = X_l.mean(axis=0, keepdims=True)
        sig = X_l.std(axis=0, keepdims=True) + 1e-8
        X_l_n  = (X_l  - mu) / sig
        X_u_n  = (X_u  - mu) / sig
        X_te_n = (X_te - mu) / sig

        # Grid search semplificato (subset di config per velocità)
        best_val, best_cfg = 0.0, None
        for xi in XI_LIST:
            for alpha in ALPHA_LIST[:3]:   # subset per velocità
                for f_dim in [20, 40]:
                    try:
                        Fu, M = dhslp_subject(X_l_n, Y_l_oh, X_u_n,
                                              f_dim=f_dim, xi=xi,
                                              alpha=alpha, beta=1.0)
                        Y_u    = RAW12_LABELS[mask_va]
                        val_b  = balanced_accuracy_score(Y_u, Fu.argmax(axis=1))
                        if val_b > best_val:
                            best_val = val_b
                            best_cfg = (xi, alpha, 1.0, f_dim)
                    except Exception as e:
                        log.warning(f'  P{sid:03d} xi={xi} a={alpha}: {e}')

        if best_cfg is None:
            log.warning(f'P{sid:03d}: nessuna config valida')
            continue

        xi, alpha, beta, f_dim = best_cfg
        Fu, M = dhslp_subject(X_l_n, Y_l_oh, X_u_n,
                              f_dim=f_dim, xi=xi, alpha=alpha, beta=beta)
        Y_pred_te = classify_test(X_l_n, Y_l, X_te_n, M, k=1)
        test_bacc = balanced_accuracy_score(Y_te, Y_pred_te)
        val_bacc  = best_val

        SUBJECT_RESULTS_15[sid] = {
            'val_bacc': val_bacc, 'test_bacc': test_bacc,
            'best_cfg': best_cfg, 'preds': Y_pred_te, 'gt': Y_te
        }
        log.info(f'P{sid:03d}: val={val_bacc:.4f}  test={test_bacc:.4f}  cfg=xi={xi} a={alpha} f={f_dim}')

    np.savez(FEAT_CACHE2, results=np.array(SUBJECT_RESULTS_15, dtype=object))
    log.info(f'Risultati salvati in {FEAT_CACHE2}')

if SUBJECT_RESULTS_15:
    chance = 1/N_CLASSES
    test_baccs = [r['test_bacc'] for r in SUBJECT_RESULTS_15.values()]
    print(f'\n── DHSLP (Li et al.) — {len(SUBJECT_RESULTS_15)} soggetti ──')
    print(f'Mean test bAcc = {np.mean(test_baccs):.4f}')
    print(f'Std  test bAcc = {np.std(test_baccs):.4f}')
    print(f'% sopra chance = {(np.array(test_baccs)>chance).mean()*100:.1f}%')
    print(f'Chance = {chance:.4f}')


## §6 — Risultati + W&B Logging

In [ ]:
if not SUBJECT_RESULTS_15:
    print('[INFO] Nessun risultato — esegui prima §5.')
else:
    chance = 1 / N_CLASSES
    rows = []
    for sid, res in SUBJECT_RESULTS_15.items():
        xi, alpha, beta, f_dim = res['best_cfg']
        rows.append({'Subject': f'P{sid:03d}', 'val_bacc': round(res['val_bacc'],4),
                     'test_bacc': round(res['test_bacc'],4),
                     'xi': xi, 'alpha': alpha, 'f_dim': f_dim})

    df15 = pd.DataFrame(rows).sort_values('test_bacc', ascending=False).reset_index(drop=True)
    df15.to_csv(FIG_DIR / 'eeg15_subject_ranking.csv', index=False)
    print(df15.to_string(index=False))

    # Bar chart per soggetto
    fig, ax = plt.subplots(figsize=(max(10, len(df15)*0.5), 5))
    colors = ['#2C7BB6' if v > chance else '#D7191C' for v in df15['test_bacc']]
    ax.bar(df15['Subject'], df15['test_bacc'], color=colors, alpha=0.9)
    ax.axhline(chance, color='black', linestyle='--', linewidth=1.5, label=f'Chance ({chance:.0%})')
    ax.set_xlabel('Soggetto'); ax.set_ylabel('Balanced Accuracy (sess.5)')
    ax.set_title(f'DHSLP (Li et al. 2025) — Per-Subject  Mean={np.mean(df15.test_bacc):.4f}')
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=90, fontsize=7)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'eeg15_per_subject.png', dpi=150); plt.show()

    # W&B
    wandb.login()
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT,
                     name=f'eeg15_DHSLP_{CLUSTER_SCHEME}',
                     config=dict(
                         notebook='EEG_15', model='DHSLP_LiEtAl2025',
                         n_classes=N_CLASSES, cluster_scheme=CLUSTER_SCHEME,
                         n_channels=N_CHANNELS, gamma_low=GAMMA_LOW, gamma_high=GAMMA_HIGH,
                         n_temp_features=N_TEMP_FEATURES, n_select_feat=N_SELECT_FEAT,
                         top_features=[FEAT_NAMES_12[i] for i in TOP_FEAT_IDX],
                         xi_list=XI_LIST, alpha_list=ALPHA_LIST, f_dim_list=F_DIM_LIST,
                         train_sessions=TRAIN_SESSIONS, val_sessions=VAL_SESSIONS,
                         test_sessions=TEST_SESSIONS, max_iter=MAX_ITER,
                         n_subjects=len(SUBJECT_RESULTS_15),
                     ),
                     reinit='finish_previous',
                     settings=wandb.Settings(start_method='thread'))
    run.log({'test/mean_bacc': float(df15.test_bacc.mean()),
             'test/std_bacc':  float(df15.test_bacc.std()),
             'test/pct_above_chance': float((df15.test_bacc > chance).mean()),
             'val/mean_bacc':  float(df15.val_bacc.mean())})
    run.summary['test_mean_bacc'] = float(df15.test_bacc.mean())
    run.finish()
    log.info('W&B completato')
